# This notebook is to sort by channel and then cluster each channel

**USE Cleaned Data!**

In [ ]:
from treeview_gen import df_to_treeview
import pandas as pd
import os
import re
import shutil
from time import sleep

regex_df = pd.read_csv("/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned_renamed/HistDiff_results_24h_bmal1ref_REDONE_WELL-DROPPED_FeatRed.csv", index_col=0, nrows=1)
pattern = r'Region (.+?) \(global\)'
channels_found = set()
for col in regex_df.columns:
    found = re.search(pattern, col)
    if found:
        channels_found.add(found.group(1))

print(channels_found)


In [ ]:
def gen_treeview_channel(df: pd.DataFrame, channel_name: str, out_dir: str, sortIndex: bool = True) -> list[str]:
    """Generates a treeviewe file set for a given channel

    Args:
        df (pd.DataFrame): channel data frame
        channel_name (str): the name of the channel
        out_dir (str): the file path where the file will be outputted to (this creates a folder)
    """
    
    out_dir_channel = os.path.join(out_dir, channel_name)
    
    if os.path.exists(out_dir_channel):
        shutil.rmtree(out_dir_channel)
        sleep(0.5)
    os.makedirs(out_dir_channel, exist_ok=True)
    
    out_dir_channel = os.path.join(out_dir_channel, f"{channel_name.replace("/","_")}_treeview")
    
    df_to_treeview(df=df, outFile=out_dir_channel, sortIndex=sortIndex, row_cluster=False)
    
    cdt_path = f"{out_dir_channel}.cdt"
    return read_tv_col_order(cdt_path)


def read_tv_col_order(file_path: str) -> list[str]:
    col_df = pd.read_csv(file_path, index_col=[0,1,2], sep="\t")
    return col_df.columns.tolist()

In [ ]:
test_cdt = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/HistDiff_results_24h_bmal1ref_FeatRed_WELL-DROPPED/Alexa 488 shortened/Alexa 488 shortened_treeview.cdt"
cdt_df = pd.read_csv(test_cdt, sep="\t", index_col=[0,1,2])

print(cdt_df.columns.tolist())

## Final Loop

In [ ]:
cleaned_file_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned"
out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted"

for file in os.listdir(cleaned_file_dir):
    curr_file = os.path.join(cleaned_file_dir, file)
    print(curr_file)
    raw_name = os.path.splitext(file)[0]
    print(raw_name)
    
    raw_out_dir = os.path.join(out_dir, raw_name)
    print(raw_out_dir)
    
    # make a folder
    if os.path.exists(raw_out_dir):
        shutil.rmtree(raw_out_dir)
        sleep(0.5)
    os.makedirs(raw_out_dir)
    
    
    curr_df = pd.read_csv(curr_file, index_col=0)
    
    # grab channels from data
    channel_df = {ch: curr_df[[col for col in curr_df.columns if ch in col]] for ch in channels_found}
    
    # grab the column orders and export out the channel clustering
    col_order = []
    for name, df in channel_df.items():
        order = gen_treeview_channel(df, name.replace("/", "_"), raw_out_dir)
        col_order.extend(order)
    
    print(col_order)
    
    reorder_df = curr_df[col_order]
    
    final_out_path = os.path.join(raw_out_dir, f"{raw_name}_SORTED-CLUSTERED_channels")
    df_to_treeview(df=reorder_df, outFile=final_out_path, sortIndex=True, row_cluster=False, col_cluster=False)

# We made an error:

- **ALL HEATMAPS MUST HAVE THE SAME COLUMN ORDERINGS AND CLUSTERING**
- This means we need to do the following:
1. Combine all the datasets together
2. Split channels
3. cluster columns of combined but channel split dataset
4. retrieve ordering
5. sort ids and apply ordering

In [ ]:
cleaned_file_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned_renamed"
out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted"

In [ ]:
# lets remove the orginal channel sorted folder
def clean_folder(path: str) -> None:
    if os.path.exists(path):
        shutil.rmtree(path)
        sleep(0.5)
    os.makedirs(path, exist_ok=True)

In [ ]:
clean_folder(out_dir)

In [ ]:
# combine data sets
renamed_datasets = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned_renamed"

dfs = []
for file in os.listdir(renamed_datasets):
    curr_file = os.path.join(renamed_datasets, file)
    curr_df = pd.read_csv(curr_file, index_col=0)
    dfs.append(curr_df)

concat_df = pd.concat(dfs)
display(concat_df)
print(concat_df.shape)

concat_df.to_csv('/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/combined_data.csv')

In [71]:
# sort, cluster, and grab/create final row ordering

channel_df = {ch: concat_df[[col for col in concat_df.columns if ch in col]] for ch in channels_found}

channel_out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/"

# if os.path.exists(channel_out_dir):
#     shutil.rmtree(channel_out_dir)
#     sleep(0.5)
# os.makedirs(channel_out_dir, exist_ok=True)

final_order = list()
for name, df in channel_df.items():
    print(name)
    print(f"Shape: {df.shape}\n")
    
    fix_name = name.replace("/", "_")
    # curr_channel_out = os.path.join(channel_out_dir, fix_name)
    # clean_folder(curr_channel_out)
    
    # file_out = os.path.join(curr_channel_out, f"{fix_name}_CHANNEL-SORTED")
    order = gen_treeview_channel(df=df, channel_name=fix_name, out_dir=channel_out_dir, sortIndex=False)
    final_order.extend(order)

    
print(len(final_order))
print(final_order)    
    
    

TMR/AF555 Rabbitts
Shape: (528, 162)

HOECHST 33342 expanded
Shape: (528, 205)

MitoTracker Deep Red
Shape: (528, 225)

SYTO14 Rabbitts
Shape: (528, 151)

Alexa 488 shortened
Shape: (528, 131)

874
['Nuclei - Cell Region TMR/AF555 Rabbitts (global) Symmetry 05', 'Nuclei - Cytoplasm Region TMR/AF555 Rabbitts (global) Symmetry 15', 'Nuclei - Cell Region TMR/AF555 Rabbitts (global) Symmetry 15', 'Nuclei - Membrane Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Ridge', 'Nuclei - Membrane Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Spot', 'Nuclei - Membrane Region TMR/AF555 Rabbitts (global) Symmetry 14 SER-Dark', 'Nuclei - Membrane Region TMR/AF555 Rabbitts (global) Symmetry 04 SER-Dark', 'Nuclei - Cytoplasm Region TMR/AF555 Rabbitts (global) Symmetry 02 SER-Dark', 'Nuclei - Ring Region TMR/AF555 Rabbitts (global) Symmetry 13 SER-Dark', 'Nuclei - Membrane Region TMR/AF555 Rabbitts (global) Symmetry 03 SER-Hole', 'Nuclei - Nucleus Region TMR/AF555 Rabb

## Final loop

In [ ]:
out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/combined_dataset"
concat_df = concat_df[final_order]
display(concat_df)
df_to_treeview(df=concat_df, outFile=out_dir, row_cluster=False, col_cluster=False, sortIndex=True)

In [ ]:
out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/feat_red_channel_sorted/"
clean_folder(out_dir)
for file in os.listdir(cleaned_file_dir):
    curr_file = os.path.join(cleaned_file_dir, file)
    
    curr_df = pd.read_csv(curr_file, index_col=0)
    curr_df = curr_df[final_order]
    out_file = os.path.splitext(file)[0]
    
    df_to_treeview(df=curr_df, outFile=os.path.join(out_dir, out_file), row_cluster=False, col_cluster=False, sortIndex=True)
    

# Split by bmal1 and per2 

## We need the following:

1. all plates/timepoints and treatment in one heatmap (but separate heatmap for Bmal1 and separate heatmap for Per2) - this one is probably the one you want to use for clustering features within channel and using that order in all other heatmaps
2. 'no treatment' heatmap (one for Bmal1 and one for Per2) - only no treatment rows but all timepoints in one heatmap 
3. 'Dex heatmap (one for Bmal1 and one for Per2) - only Dex rows but all timepoints in one heatmap 
4. 'PF' heatmap (one for Bmal1 and one for Per2) - only PF rows but all timepoints in one heatmap 
5. '24h' heatmap (one for Bmal1 and one for Per2) - only '24h' plate but all treatments in one heatmap 
6. '66h' heatmap (one for Bmal1 and one for Per2) - only '66h' plate but all treatments in one heatmap 
7. '72h' heatmap (one for Bmal1 and one for Per2) - only '72h' plate but all treatments in one heatmap


*Note: BMAL1 files use BMAL1ref!*

bmal1

In [87]:
bmal_out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/bmal1"
in_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned_renamed"

clean_folder(bmal_out_dir)

bmal1_files = [os.path.join(in_dir, file) for file in os.listdir(in_dir) if 'bmal1ref' in file]

concat_df = [pd.read_csv(file, index_col=0) for file in bmal1_files]
concat_df = pd.concat(concat_df, axis=0)
concat_df = concat_df[concat_df.index.str.contains("_bmal1._.", regex=False)]

# display(concat_df)

# grabbing channels and clusters

channel_dfs = {channel: concat_df[[col for col in concat_df.columns if channel in col]] for channel in channels_found}

bmal_clus_order = list()
channel_out_dir = os.path.join(bmal_out_dir, "channels")
for name, df in channel_dfs.items():
    print(name, df.shape)
    
    fix_name = name.replace("/", "_")
    order = gen_treeview_channel(df=df, channel_name=fix_name, out_dir=channel_out_dir)
    bmal_clus_order.extend(order)

print(len(bmal_clus_order))
print(bmal_clus_order)

all_timepoints = concat_df[bmal_clus_order]

# combined timepoints
df_to_treeview(df=all_timepoints, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(bmal_out_dir, "combined_timepoints"))

# no treatment
no_treatment = all_timepoints[all_timepoints.index.str.contains("no_treatment_", regex=False)]

try:
    df_to_treeview(df=no_treatment, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(bmal_out_dir, "no_treatment"))
except:
    pass

# Dex
dex = all_timepoints[all_timepoints.index.str.contains("Dex ", regex=False)]

try:
    df_to_treeview(df=dex, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(bmal_out_dir, "Dex"))
except:
    pass

#PF

pf = all_timepoints[all_timepoints.index.str.contains("PF_", regex=False)]

try:
    df_to_treeview(df=pf, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(bmal_out_dir, "PF"))
except:
    pass

#24h

h24 = all_timepoints[all_timepoints.index.str.contains("._.24h._.", regex=False)]

try:
    df_to_treeview(df=h24, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(bmal_out_dir, "24h"))
except:
    pass

#66h

h66 = all_timepoints[all_timepoints.index.str.contains("._.66h._.", regex=False)]

try:
    df_to_treeview(df=h66, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(bmal_out_dir, "66h"))
except:
    pass

#72h

h72 = all_timepoints[all_timepoints.index.str.contains("._.72h._.", regex=False)]

try:
    df_to_treeview(df=h72, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(bmal_out_dir, "72h"))
except:
    pass

TMR/AF555 Rabbitts (198, 162)
HOECHST 33342 expanded (198, 205)
MitoTracker Deep Red (198, 225)
SYTO14 Rabbitts (198, 151)
Alexa 488 shortened (198, 131)
874
['Nuclei - Membrane Region TMR/AF555 Rabbitts (global) Threshold Compactness 60% SER-Dark', 'Nuclei - Cell Region TMR/AF555 Rabbitts (global) Profile 2/5', 'Nuclei - Nucleus Region TMR/AF555 Rabbitts (global) Gabor Min 2 px w2', 'Nuclei - Intensity Membrane Region TMR/AF555 Rabbitts (global) Contrast', 'Nuclei - Nucleus Region TMR/AF555 Rabbitts (global) Threshold Compactness 60% SER-Spot', 'Nuclei - Cell Region TMR/AF555 Rabbitts (global) Threshold Compactness 30%', 'Nuclei - Nucleus Region TMR/AF555 Rabbitts (global) Radial Mean Ratio SP-Filter', 'Nuclei - Nucleus Region TMR/AF555 Rabbitts (global) Symmetry 05 SER-Spot', 'Nuclei - Nucleus Region TMR/AF555 Rabbitts (global) Symmetry 15 SER-Spot', 'Nuclei - Intensity Membrane Region TMR/AF555 Rabbitts (global) Median', 'Nuclei - Nucleus Region TMR/AF555 Rabbitts (global) Threshold

Per2

In [88]:
per2_out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/per2"
in_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned_renamed"

clean_folder(per2_out_dir)

per2_files = [os.path.join(in_dir, file) for file in os.listdir(in_dir) if 'per2ref' in file]

concat_df = [pd.read_csv(file, index_col=0) for file in per2_files]
concat_df = pd.concat(concat_df, axis=0)
concat_df = concat_df[concat_df.index.str.contains("_per2._.", regex=False)]

# display(concat_df)

# grabbing channels and clusters

channel_dfs = {channel: concat_df[[col for col in concat_df.columns if channel in col]] for channel in channels_found}

per2_clus_order = list()
channel_out_dir = os.path.join(per2_out_dir, "channels")
for name, df in channel_dfs.items():
    print(name, df.shape)
    
    fix_name = name.replace("/", "_")
    order = gen_treeview_channel(df=df, channel_name=fix_name, out_dir=channel_out_dir)
    per2_clus_order.extend(order)

print(len(per2_clus_order))
print(per2_clus_order)

all_timepoints = concat_df[per2_clus_order]

# combined timepoints
df_to_treeview(df=all_timepoints, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(per2_out_dir, "combined_timepoints"))

# no treatment
no_treatment = all_timepoints[all_timepoints.index.str.contains("no_treatment_", regex=False)]

try:
    df_to_treeview(df=no_treatment, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(per2_out_dir, "no_treatment"))
except:
    pass

# Dex
dex = all_timepoints[all_timepoints.index.str.contains("Dex ", regex=False)]

try:
    df_to_treeview(df=dex, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(per2_out_dir, "Dex"))
except:
    pass

#PF

pf = all_timepoints[all_timepoints.index.str.contains("PF_", regex=False)]

try:
    df_to_treeview(df=pf, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(per2_out_dir, "PF"))
except:
    pass

#24h

h24 = all_timepoints[all_timepoints.index.str.contains("._.24h._.", regex=False)]

try:
    df_to_treeview(df=h24, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(per2_out_dir, "24h"))
except:
    pass

#66h

h66 = all_timepoints[all_timepoints.index.str.contains("._.66h._.", regex=False)]

try:
    df_to_treeview(df=h66, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(per2_out_dir, "66h"))
except:
    pass

#72h

h72 = all_timepoints[all_timepoints.index.str.contains("._.72h._.", regex=False)]

try:
    df_to_treeview(df=h72, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(per2_out_dir, "72h"))
except:
    pass

TMR/AF555 Rabbitts (264, 162)
HOECHST 33342 expanded (264, 205)
MitoTracker Deep Red (264, 225)
SYTO14 Rabbitts (264, 151)
Alexa 488 shortened (264, 131)
874
['Nuclei - Cell Region TMR/AF555 Rabbitts (global) Profile 2/5', 'Nuclei - Ring Region TMR/AF555 Rabbitts (global) Profile 5/5 SER-Hole', 'Nuclei - Intensity Cytoplasm Region TMR/AF555 Rabbitts (global) Contrast', 'Nuclei - Intensity Membrane Region TMR/AF555 Rabbitts (global) Contrast', 'Nuclei - Membrane Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Ridge', 'Nuclei - Membrane Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Spot', 'Nuclei - Cytoplasm Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Saddle', 'Nuclei - Cytoplasm Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Valley', 'Nuclei - Cytoplasm Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Edge', 'Nuclei - Nucleus Region TMR/AF555 Rabbitts (global) Symmetry 03 SER-Edge', 'Nuclei - Nucl